In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer,  StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
tqdm.pandas(desc="Processing Rows")
import os

import gc
import time
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
import ast
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    r2_score
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
hf_token = "os.environ['HF_TOKEN']" 

/apps/common/software/Miniforge3/25.11.0-1-jupyter-base/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load in Model

In [2]:
!ls /common/data/models/

bartowski--OpenGVLab_InternVL3_5-30B-A3B-GGUF
bartowski--OpenGVLab_InternVL3_5-8B-GGUF
black-forest-labs--FLUX.1-dev
black-forest-labs--FLUX.2-dev
city96--Qwen-Image-gguf--BF16
cyankiwi--GLM-4.6V-AWQ-4bit
cyankiwi--MiniMax-M2.7-AWQ-4bit
cyankiwi--MiniMax-M3-AWQ-INT4
deepseek-ai--DeepSeek-R1-0528-Qwen3-8B
deepseek-ai--DeepSeek-R1-Distill-Llama-70B
deepseek-ai--DeepSeek-R1-Distill-Llama-8B
deepseek-ai--DeepSeek-R1-Distill-Qwen-14B
deepseek-ai--DeepSeek-R1-Distill-Qwen-32B
deepseek-ai--DeepSeek-V3.2
deepseek-ai--DeepSeek-V4-Flash
ggml-org--GLM-4.5V-GGUF--Q4_K_M
ggml-org--gpt-oss-120b-GGUF
ggml-org--gpt-oss-20b-GGUF
google--gemma-3-12b-it
google--gemma-3-1b-it
google--gemma-3-27b-it
google--gemma-3-4b-it
google--gemma-4-12B
google--gemma-4-12B-it
google--gemma-4-12B-it-assistant
google--gemma-4-26B-A4B
google--gemma-4-26B-A4B-it
google--gemma-4-26B-A4B-it-assistant
google--gemma-4-31B
google--gemma-4-31B-it
google--gemma-4-31B-it-assistant
google--gemma-4-E2B
google--gemma-4-E2B-it
google-

In [3]:
model_id = "/common/data/models/qwen--Qwen3-4B"
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = "os.environ['HF_TOKEN']" 

tokenizer = AutoTokenizer.from_pretrained(model_id,token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 18071.66it/s]


### Data Reading and formatting

In [4]:

df=pd.read_csv("Datasets/World_Ecological_2015_Metadata.csv")

columns = [
    'geonameid', 'name', 'asciiname', 'alternatenames', 
    'latitude', 'longitude', 'feature_class', 'feature_code', 
    'country_code', 'cc2', 'admin1', 'admin2', 'admin3', 'admin4', 
    'population', 'elevation', 'dem', 'timezone', 'modification_date'
]

df_geo = pd.read_csv('Datasets/allCountries.txt', sep='\t', names=columns, low_memory=False)
df_landforms = df_geo[df_geo['feature_class'] == 'T']
combined_dataset=pd.read_csv("Datasets/Global_Named_Landforms_With_Labels.csv")
combined_dataset=combined_dataset[['name','alternatenames','feature_code','ELU_LF_Des']]


/localscratch/471331/ipykernel_1996461/2529600610.py:12: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_dataset=pd.read_csv("Datasets/Global_Named_Landforms_With_Labels.csv")


In [5]:
exact_6_class_map = {
    # 1: Plains
    'PLN': 1, 'PLNS': 1, 'FLT': 1, 'PANS': 1,
    
    # 2: Hills and Low Tablelands
    'HLL': 2, 'HLLS': 2, 'MND': 2, 'KNOB': 2,
    
    # 3: High Tablelands
    'PT': 3, 'PLAT': 3, 'MESA': 3, 'BUTT': 3,
    
    # 4: Mountains
    'RDG': 4, 'RDGS': 4, 'SADL': 4, 'DUNE': 4,
    
    # 5: Widely Spaced Mountains
    'MT': 5, 'MTS': 5, 'PK': 5, 'PKS': 5, 'CONE': 5, 'VOLC': 5, 'NUP': 5, 'RK': 5,
    
    # 7: Depressions or Basins
    'BSN': 7, 'BSNS': 7, 'DPR': 7, 'SINK': 7, 'CRQS': 7, 'CRATER': 7
}

combined_dataset['macro_class_id'] = combined_dataset['feature_code'].map(exact_6_class_map)
combined_dataset = combined_dataset.dropna(subset=['macro_class_id'])
combined_dataset['macro_class_id'] = combined_dataset['macro_class_id'].astype(int)
combined_dataset

,name,alternatenames,feature_code,ELU_LF_Des,macro_class_id
0,Roc Meler,"Roc Mele,Roc Meler,Roc Mélé",PK,Mountains,5
1,Pic de les Abelletes,"Pic de la Font-Negre,Pic de la Font-Nègre,Pic de les Abelletes",PK,Mountains,5
2,Pic de Meners,NaN,PK,Mountains,5
4,Roc de Port Dret,NaN,PK,Mountains,5
6,Roc del Xeig,NaN,RK,Mountains,5
...,...,...,...,...,...
1834929,South Rock,"Danzhu,Danzhu Shi,Danzhushi,South Rock,Tan-chu Shih,dan zhu,dan zhu shi,单柱,单柱石",RK,NaN,5
1834942,Whale Rock (historical),NaN,RK,NaN,5
1834949,Echo Point - Three Sisters,NaN,MT,NaN,5
1834953,Ipil Hill,"Ipil Hill,Ipil Seamount",HLL,NaN,2


In [6]:
combined_dataset=combined_dataset[~combined_dataset['macro_class_id'].isna()]
combined_dataset=combined_dataset[['name','macro_class_id']]
combined_dataset.columns=['Bridges Full Name','Topographic']
combined_dataset = combined_dataset.groupby('Bridges Full Name')['Topographic'].apply(lambda x: list(set(x))).reset_index()
ascii_mask = combined_dataset['Bridges Full Name'].astype(str).str.contains(r'^[\x00-\x7F]+$')
combined_dataset = combined_dataset[ascii_mask].reset_index(drop=True)
combined_dataset

,Bridges Full Name,Topographic
0,'Adade Yus Mountain,[5]
1,'Aqabat Nikid,[5]
2,'Elb Chouf,[4]
3,'Elb ech Chaoufa,[4]
4,'S Airde Beinn,[2]
...,...,...
618599,wzgorze na Katarynkach,[2]
618600,wzgorze nad Tuchlinkiem,[2]
618601,yak Gawa,[5]
618602,ytre Midtberget,[2]


### Test Train split

In [7]:
def label_to_list(x):
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]

    if isinstance(x, (list, tuple)):
        return [int(v) for v in x]

    if isinstance(x, str):
        x = x.strip()
        if x.startswith("["):
            return [int(v) for v in ast.literal_eval(x)]
        return [int(float(x))]

    return [int(x)]


X = combined_dataset["Bridges Full Name"].astype(str)
y_labels = combined_dataset["Topographic"].apply(label_to_list)

# optional: stratify by exact label-combination string
y_combo = y_labels.apply(lambda z: tuple(sorted(z)))

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X,
    y_labels,
    test_size=0.20,
    stratify=None,
    random_state=42
)
MAX_TRAIN = 100000
MAX_TEST = 20000

train_combo = y_train_full.apply(lambda z: tuple(sorted(z)))
test_combo = y_test_full.apply(lambda z: tuple(sorted(z)))

X_train_list, _, y_train_labels, _ = train_test_split(
    X_train_full.tolist(),
    y_train_full.tolist(),
    train_size=min(MAX_TRAIN, len(X_train_full)),
    stratify=None,
    random_state=42
)

X_test_list, _, y_test_labels, _ = train_test_split(
    X_test_full.tolist(),
    y_test_full.tolist(),
    train_size=min(MAX_TEST, len(X_test_full)),
    stratify=None,
    random_state=42
)

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform(y_train_labels)
Y_test = mlb.transform(y_test_labels)

print("Classes:", mlb.classes_)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

Classes: [1 2 3 4 5 7]
Y_train shape: (100000, 6)
Y_test shape: (20000, 6)


In [8]:


# BATCH_SIZE = 1024
# MAX_LEN = 64

# num_blocks = len(model.model.layers)
# print("Transformer blocks:", num_blocks)
# HIDDEN_STATE_LAYERS = [i for i in range(0,num_blocks+1,4)]

# print("Using hidden_state indices:", HIDDEN_STATE_LAYERS)

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# tokenizer.padding_side = "right"

# model.eval()
# model.config.use_cache = False

# if hasattr(model, "gradient_checkpointing_disable"):
#     model.gradient_checkpointing_disable()

# train_loader = DataLoader(
#     X_train_list,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=False
# )

# test_loader = DataLoader(
#     X_test_list,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=False
# )
# # first extract layers
# def extract_selected_token_states(loader, desc, hidden_state_layers):
#     chunks = {l: [] for l in hidden_state_layers}

#     with torch.inference_mode():
#         for batch in tqdm(loader, desc=desc):
#             inputs = tokenizer(
#                 list(batch),
#                 padding=True,
#                 truncation=True,
#                 max_length=MAX_LEN,
#                 return_tensors="pt"
#             ).to(model.device)
#             lengths = inputs["attention_mask"].sum(dim=1) - 1
#             batch_idx = torch.arange(inputs["input_ids"].shape[0], device=model.device)

#             with torch.inference_mode():
#                 outputs = model(
#                     **inputs,
#                     output_hidden_states=True,
#                     use_cache=False
#                 )

#             for l in hidden_state_layers:
#                 hs = outputs.hidden_states[l]
#                 tok_state = hs[batch_idx, lengths, :]
#                 chunks[l].append(tok_state.float().cpu().numpy().astype(np.float32))

#             del inputs, outputs, lengths, batch_idx, tok_state
#             gc.collect()

#     return {l: np.vstack(chunks[l]) for l in hidden_state_layers}

# start = time.perf_counter()

# print("\nExtracting train hidden states...")
# train_cache = extract_selected_token_states(train_loader, "Train", HIDDEN_STATE_LAYERS)

# print("\nExtracting test hidden states...")
# test_cache = extract_selected_token_states(test_loader, "Test", HIDDEN_STATE_LAYERS)

# print(f"\nExtraction time: {(time.perf_counter() - start) / 60:.2f} min")

# torch.cuda.empty_cache()
# gc.collect()


# acc_list = []
# f1_list = []
# precision_list = []
# recall_list = []
# r2_list = []

# best_score = -1
# best_layer = None
# best_probe = None
# best_pred = None

# for hs_idx in HIDDEN_STATE_LAYERS:
#     print(f"\n--- Linear probe on hidden_states[{hs_idx}] ---")

#     Xtr = train_cache[hs_idx]
#     Xte = test_cache[hs_idx]

#     probe = make_pipeline(
#         StandardScaler(),
#         OneVsRestClassifier(
#             SGDClassifier(
#                 loss="log_loss",
#                 penalty="l2",
#                 alpha=1e-4,
#                 max_iter=50,
#                 tol=1e-3,
#                 random_state=42,
#                 early_stopping=False,
#                 n_jobs=-1
#             ),
#             n_jobs=-1
#         )
#     )

#     probe.fit(Xtr, Y_train)
#     pred = probe.predict(Xte)
    
#     acc = accuracy_score(Y_test, pred)
#     f1 = f1_score(Y_test, pred, average="weighted", zero_division=0)
#     precision = precision_score(Y_test, pred, average="weighted", zero_division=0)
#     recall = recall_score(Y_test, pred, average="weighted", zero_division=0)
#     r2 = r2_score(Y_test, pred)

#     acc_list.append(acc)
#     f1_list.append(f1)
#     precision_list.append(precision)
#     recall_list.append(recall)
#     r2_list.append(r2)

#     print(f"Accuracy:  {acc:.4f}")
#     print(f"F1:        {f1:.4f}")
#     print(f"Precision: {precision:.4f}")
#     print(f"Recall:    {recall:.4f}")
#     print(f"R2:        {r2:.4f}")

#     if f1 > best_score:
#         best_score = f1
#         best_layer = hs_idx
#         best_probe = probe
#         best_pred = pred.copy()
#     gc.collect()

In [9]:

torch.cuda.empty_cache()
BATCH_SIZE = 32
MAX_LEN = 64

num_blocks = len(model.model.layers)
print("Transformer blocks:", num_blocks)
HIDDEN_STATE_LAYERS = [i for i in range(0,num_blocks+1,4)]

print("Using hidden_state indices:", HIDDEN_STATE_LAYERS)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model.eval()
model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

train_loader = DataLoader(
    X_train_list,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    X_test_list,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)
# first extract layers
def extract_selected_token_states(loader, desc, hidden_state_layers):
    chunks = {l: [] for l in hidden_state_layers}

    with torch.inference_mode():
        for batch in tqdm(loader, desc=desc):
            inputs = tokenizer(
                list(batch),
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(model.device)
            lengths = inputs["attention_mask"].sum(dim=1) - 1
            batch_idx = torch.arange(inputs["input_ids"].shape[0], device=model.device)

            with torch.inference_mode():
                outputs = model(
                    **inputs,
                    output_hidden_states=True,
                    use_cache=False
                )

            for l in hidden_state_layers:
                hs = outputs.hidden_states[l]
                tok_state = hs[batch_idx, lengths, :]
                chunks[l].append(tok_state.float().cpu().numpy().astype(np.float32))

            del inputs, outputs, lengths, batch_idx, tok_state
            gc.collect()

    return {l: np.vstack(chunks[l]) for l in hidden_state_layers}

start = time.perf_counter()

print("\nExtracting train hidden states...")
train_cache = extract_selected_token_states(train_loader, "Train", HIDDEN_STATE_LAYERS)

print("\nExtracting test hidden states...")
test_cache = extract_selected_token_states(test_loader, "Test", HIDDEN_STATE_LAYERS)

print(f"\nExtraction time: {(time.perf_counter() - start) / 60:.2f} min")

torch.cuda.empty_cache()
gc.collect()


acc_list = []
f1_list = []
precision_list = []
recall_list = []
r2_list = []

best_score = -1
best_layer = None
best_probe = None
best_pred = None

for hs_idx in HIDDEN_STATE_LAYERS:
    print(f"\n--- Linear probe on hidden_states[{hs_idx}] ---")

    Xtr = train_cache[hs_idx]
    Xte = test_cache[hs_idx]

    probe = make_pipeline(
        OneVsRestClassifier(
            ExtraTreesClassifier(
                n_estimators=200,
                n_jobs=-1,
                random_state=42
            )
        )
    )

    probe.fit(Xtr, Y_train)
    pred = probe.predict(Xte)
    
    acc = accuracy_score(Y_test, pred)
    f1 = f1_score(Y_test, pred, average="weighted", zero_division=0)
    precision = precision_score(Y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(Y_test, pred, average="weighted", zero_division=0)
    r2 = r2_score(Y_test, pred)

    acc_list.append(acc)
    f1_list.append(f1)
    precision_list.append(precision)
    recall_list.append(recall)
    r2_list.append(r2)

    print(f"Accuracy:  {acc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"R2:        {r2:.4f}")

    if f1 > best_score:
        best_score = f1
        best_layer = hs_idx
        best_probe = probe
        best_pred = pred.copy()
    gc.collect()
# Save all lists into one text file
output_filename = "linear_probe_metrics_NL_qwen4b.txt"

with open(output_filename, "w") as f:
    f.write(f"Layers:    {HIDDEN_STATE_LAYERS}\n")
    f.write(f"Accuracy:  {acc_list}\n")
    f.write(f"F1:        {f1_list}\n")
    f.write(f"Precision: {precision_list}\n")
    f.write(f"Recall:    {recall_list}\n")
    f.write(f"R2:        {r2_list}\n")

print(f"Metrics saved to {output_filename}")

Transformer blocks: 36
Using hidden_state indices: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36]

Extracting train hidden states...


Train: 100%|██████████| 3125/3125 [52:12<00:00,  1.00s/it]  



Extracting test hidden states...


Test: 100%|██████████| 625/625 [07:44<00:00,  1.35it/s]



Extraction time: 60.02 min

--- Linear probe on hidden_states[0] ---


NameError: name 'ExtraTreesClassifier' is not defined

In [ ]:
depth = np.array(HIDDEN_STATE_LAYERS) / num_blocks

plt.figure(figsize=(9, 5))

plt.plot(depth, acc_list, marker="o", label="Accuracy")
plt.plot(depth, f1_list, marker="s", label="F1")
plt.plot(depth, precision_list, marker="^", label="Precision")
plt.plot(depth, recall_list, marker="D", label="Recall")

plt.xlabel("Normalized layer depth")
plt.ylabel("Score")
plt.title("Linear Probe: Landform Type from Bridge Name")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# # Save all lists into one text file
# output_filename = "linear_probe_metrics_qwen4b.txt"

# with open(output_filename, "w") as f:
#     f.write(f"Layers:    {HIDDEN_STATE_LAYERS}\n")
#     f.write(f"Accuracy:  {acc_list}\n")
#     f.write(f"F1:        {f1_list}\n")
#     f.write(f"Precision: {precision_list}\n")
#     f.write(f"Recall:    {recall_list}\n")
#     f.write(f"R2:        {r2_list}\n")

# print(f"Metrics saved to {output_filename}")